In [2]:
```python
import torch
# PyTorch: used for tensors, neural networks, and inference

import torchaudio
# PyTorch's audio library: used to load and resample audio

from transformers import WavLMModel, Wav2Vec2FeatureExtractor
# WavLMModel = pretrained WavLM neural network
# Wav2Vec2FeatureExtractor = prepares raw audio for WavLM


# ============================================================
# SPEAKER ENCODER
# ============================================================

class SpeakerEncoder:

    def __init__(self):

        print(
            "Loading pretrained speaker encoder..."
        )
        # Inform us that the pretrained model is being loaded


        # ----------------------------------------------------
        # LOAD AUDIO FEATURE EXTRACTOR
        # ----------------------------------------------------

        self.processor = Wav2Vec2FeatureExtractor.from_pretrained(
            "microsoft/wavlm-base-plus"
        )
        # Load Microsoft's preprocessing configuration
        # for the WavLM Base Plus model
        #
        # It prepares the raw waveform in the format
        # expected by WavLM


        # ----------------------------------------------------
        # LOAD PRETRAINED WavLM MODEL
        # ----------------------------------------------------

        self.model = WavLMModel.from_pretrained(
            "microsoft/wavlm-base-plus"
        )
        # Download/load the pretrained WavLM Base Plus model
        #
        # WavLM learns useful representations of speech,
        # including information related to:
        #
        # - speaker characteristics
        # - speech patterns
        # - acoustic characteristics
        # - phonetic information


        # ----------------------------------------------------
        # EVALUATION MODE
        # ----------------------------------------------------

        self.model.eval()
        # Put the model into evaluation/inference mode
        #
        # This disables training-specific behavior such as
        # dropout


        print(
            "Speaker encoder loaded successfully."
        )


    # ========================================================
    # CREATE SPEAKER EMBEDDING
    # ========================================================

    def encode(self, wav_path):

        # ----------------------------------------------------
        # LOAD AUDIO
        # ----------------------------------------------------

        waveform, sample_rate = torchaudio.load(
            wav_path
        )
        # Load the WAV file
        #
        # waveform:
        #   audio samples represented as a PyTorch tensor
        #
        # sample_rate:
        #   original sampling rate of the audio


        # ----------------------------------------------------
        # CONVERT STEREO → MONO
        # ----------------------------------------------------

        if waveform.shape[0] > 1:

            waveform = waveform.mean(
                dim=0,
                keepdim=True
            )

        # waveform.shape[0] represents the number of
        # audio channels.
        #
        # Example:
        #
        # Stereo:
        #   [Left channel, Right channel]
        #
        # Mono:
        #   [single channel]
        #
        # WavLM can work with a single-channel waveform,
        # so stereo audio is converted to mono.


        # ----------------------------------------------------
        # RESAMPLE TO 16 kHz
        # ----------------------------------------------------

        if sample_rate != 16000:

            resampler = torchaudio.transforms.Resample(
                orig_freq=sample_rate,
                new_freq=16000
            )
            # Create a resampling operation
            # to convert the audio to 16 kHz


            waveform = resampler(
                waveform
            )
            # Apply the resampling


        # ----------------------------------------------------
        # REMOVE CHANNEL DIMENSION
        # ----------------------------------------------------

        waveform = waveform.squeeze(0)
        # Convert:
        #
        # [1, number_of_samples]
        #
        # into:
        #
        # [number_of_samples]
        #
        # This is the shape expected by the feature extractor


        # ----------------------------------------------------
        # PREPARE AUDIO FOR WavLM
        # ----------------------------------------------------

        inputs = self.processor(
            waveform.numpy(),

            sampling_rate=16000,

            return_tensors="pt"
        )
        # Convert the audio into the format expected by
        # the Hugging Face WavLM model
        #
        # return_tensors="pt"
        # means return PyTorch tensors


        # ====================================================
        # RUN WavLM
        # ====================================================

        with torch.no_grad():

            outputs = self.model(
                inputs.input_values
            )

        # torch.no_grad() tells PyTorch that we are only
        # performing inference.
        #
        # We are NOT training WavLM here.
        #
        # This reduces memory usage and makes inference faster.


        # ====================================================
        # MEAN POOL ACROSS TIME
        # ====================================================

        embedding = outputs.last_hidden_state.mean(
            dim=1
        )

        # WavLM produces a representation for many
        # points/frames across the audio.
        #
        # Shape is approximately:
        #
        # [batch_size, time_steps, hidden_size]
        #
        # Mean-pooling across dim=1 combines all time
        # frames into ONE fixed-size representation.
        #
        # Result:
        #
        # [batch_size, hidden_size]


        # ====================================================
        # L2 NORMALIZATION
        # ====================================================

        embedding = torch.nn.functional.normalize(
            embedding,
            p=2,
            dim=1
        )

        # Normalize the embedding to have L2 norm = 1.
        #
        # This makes embeddings easier to compare using
        # similarity measures such as cosine similarity.


        # ====================================================
        # RETURN SINGLE EMBEDDING
        # ====================================================

        return embedding.squeeze(0)

        # Remove the batch dimension.
        #
        # Example:
        #
        # [1, 768]
        #
        # becomes:
        #
        # [768]
        #
        # The returned vector represents the audio.


# ============================================================
# TEST THE ENCODER
# ============================================================

if __name__ == "__main__":

    encoder = SpeakerEncoder()
    # Create the SpeakerEncoder object
    #
    # This loads the pretrained WavLM model.


    print(
        "Speaker encoder loaded successfully."
    )
    # Confirm that the encoder initialized correctly


/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading pretrained speaker encoder...
Speaker encoder loaded successfully.
Speaker encoder loaded successfully.
